1. IMPORT LIBRARY

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


2. LOAD DATA PREPROCESSING

In [2]:
df = pd.read_csv('processed_data.csv')
df.head()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,house_age,was_renovated,years_since_renovation,city_encoded
0,313000.0,3.0,1.50,1340,7912,1.5,0,0,3,1340,0,59,1,9,0.026739
1,342000.0,3.0,2.00,1930,11947,1.0,0,0,4,1930,0,48,0,48,0.040217
2,420000.0,3.0,2.25,2000,8030,1.0,0,0,4,1000,1000,51,0,51,0.062174
3,550000.0,4.0,2.50,1940,10500,1.0,0,0,4,1140,800,38,1,22,0.051087
4,490000.0,2.0,1.00,880,6380,1.0,0,0,3,880,0,76,1,20,0.341957


3. SEPARATE FEATURE (X) AND TARGET (Y)

In [3]:
X = df.drop(columns=['price'])
y = df['price']

X.columns.tolist()

['bedrooms',
 'bathrooms',
 'sqft_living',
 'sqft_lot',
 'floors',
 'waterfront',
 'view',
 'condition',
 'sqft_above',
 'sqft_basement',
 'house_age',
 'was_renovated',
 'years_since_renovation',
 'city_encoded']

4. SPLIT DATA TEST AND TRAIN

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train.shape, X_test.shape


((3427, 14), (857, 14))

5. FEATURE STANDARDIZATION

In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

6. DEFINITION OF MODEL COMPARISON

In [6]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
}


7. TRAIN AND EVALUATE EACH MODEL USING CROSS-VALIDATION

In [7]:
results = []

for name, model in models.items():
    if name in ['Linear Regression', 'Ridge Regression']:
        X_tr, X_te = X_train_scaled, X_test_scaled
    else:
        X_tr, X_te = X_train, X_test

    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    cv_scores = cross_val_score(model, X_tr, y_train, cv=5, scoring='r2')

    results.append({
        'model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
        'CV_R2_mean': cv_scores.mean(),
        'CV_R2_std': cv_scores.std()
    })

results_df = pd.DataFrame(results).sort_values('R2', ascending=False)
results_df


,model,MAE,RMSE,R2,CV_R2_mean,CV_R2_std
4,Gradient Boosting,95622.614321,137490.067207,0.605321,0.603832,0.027105
3,Random Forest,95023.841732,139523.575719,0.593560,0.592707,0.025531
0,Linear Regression,122692.698349,162175.390121,0.450875,0.449704,0.026778
1,Ridge Regression,122696.745095,162175.543199,0.450874,0.449710,0.026772
2,Decision Tree,136355.711365,204604.895043,0.125956,0.192246,0.089752


8. CHOOSE THE BEST MODEL AND SIMPLE TUNING (RANDOM FOREST)

In [8]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

rf = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

print('Best Params:', grid_search.best_params_)
print('Best CV R2:', grid_search.best_score_)


Best Params: {'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 200}
Best CV R2: 0.5938637188283855


9. EVALUATION THE BEST MODEL FOR DATA TEST

In [9]:
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_best)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_best))
r2 = r2_score(y_test, y_pred_best)

print(f'MAE : {mae:,.2f}')
print(f'RMSE: {rmse:,.2f}')
print(f'R2  : {r2:.4f}')


MAE : 95,342.13
RMSE: 139,950.23
R2  : 0.5911


10. SAVE MODEL AND SCALER

In [10]:
joblib.dump(best_model, 'model_rumah.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(X.columns.tolist(), 'feature_columns.pkl')

y_test.to_csv('y_test.csv', index=False)
pd.DataFrame(y_pred_best, columns=['y_pred']).to_csv('y_pred_best.csv', index=False)
X_test.to_csv('X_test.csv', index=False)

print('The Model and the Result have been saved')

The Model and the Result have been saved
